# 05 · RISC-V 静态模型验证固件

阶段 5.3：先读已完成的交叉编译证据，再了解板端自检。默认只读报告；编译成功尚不能说明板端输出正确。详细解释见 [静态固件教程](../docs/04_deployment/riscv_static_firmware.md)。

## 1. 找到项目与真实报告

`Path` 组合路径，JSON 保存构建证据。这里的 `target_run_verified=False` 必须保留到真实上板日志通过。

In [1]:
from pathlib import Path
import json, hashlib, subprocess, sys
ROOT = Path.cwd()
if not (ROOT / "MAIN.md").exists():
    ROOT = ROOT.parent
FPGA = ROOT.parent / "CNN-Tutorial-FPGA"
report = json.loads((ROOT / "docs/05_experiments/v0.4/build_report.json").read_text(encoding="utf-8"))
{key: report[key] for key in ["compiler", "isa", "abi", "units", "arena_ceiling_bytes", "arena_used_bytes", "target_run_verified"]}

{'compiler': 'riscv-none-elf-g++.exe (xPack GNU RISC-V Embedded GCC x86_64) 13.4.0',
 'isa': 'rv32im_zicsr_zifencei',
 'abi': 'ilp32',
 'units': 108,
 'arena_ceiling_bytes': 262144,
 'arena_used_bytes': None,
 'target_run_verified': False}

## 2. 模型大小与固件内存不同

模型是权重和图；固件还包含解释器、测试输入、arena、栈。GNU size 的 bss 合计包含预留栈，查看 ELF section 才能区分。

In [2]:
print(report["size_output"])
print((ROOT / "docs/05_experiments/v0.4/elf_sections.txt").read_text(encoding="utf-8"))

   text	   data	    bss	    dec	    hex	filename
  92992	  58686	2359928	2511606	 2652f6	cnn_static.elf

There are 23 section headers, starting at offset 0x3b289c:

Section Headers:
  [Nr] Name              Type            Addr     Off    Size   ES Flg Lk Inf Al
  [ 0]                   NULL            00000000 000000 000000 00      0   0  0
  [ 1] .init             PROGBITS        00001000 001000 000070 00  AX  0   0  1
  [ 2] .text             PROGBITS        00001070 001070 016ad0 00  AX  0   0  4
  [ 3] .init_array       INIT_ARRAY      00017b40 017b40 000048 04  WA  0   0  4
  [ 4] .fini_array       FINI_ARRAY      00017b88 017b88 000004 04  WA  0   0  4
  [ 5] .data             PROGBITS        00017b90 017b90 00e4f2 00  WA  0   0 16
  [ 6] .bss              NOBITS          00026090 026082 040278 00  WA  0   0 16
  [ 7] .stack            NOBITS          00066310 026082 200000 00  WA  0   0 16
  [ 8] .debug_info       PROGBITS        00000000 026082 2287c0 00      0   0  1
  [ 9] .

## 3. 核对本地 ELF

SHA-256 检查归档与实际文件是否一致；缺少本地产物时只报告缺失，不伪造验证结果。调试路径影响 ELF 哈希，因此独立重建有自己的报告。

In [3]:
run_dir = FPGA / "artifacts/cnn-static-2026-r5"
elf = run_dir / "cnn_static.elf"
if elf.exists():
    assert hashlib.sha256(elf.read_bytes()).hexdigest() == report["elf_sha256"]
    print("本地 ELF 与归档报告一致；尚未执行板端推理。")
else:
    print("本机没有该 ELF；仍可阅读已归档的构建结果。")

本地 ELF 与归档报告一致；尚未执行板端推理。


## 4. 可选重新编译

设 `REBUILD=True` 后使用真实 SDK 和只读厂商依赖，输出必须选未存在的目录。此代码不进行 FPGA 下载或 JTAG 操作。不要在用户禁止修改的原工程中运行。

In [4]:
REBUILD = False
if REBUILD:
    desktop = ROOT.parent
    vendor = desktop / "Work/FPGA Contest/demo/tinyml-main/tinyml_hello_world/Ti60F225_tinyml_helloworld/embedded_sw/sapphire_soc_tinyml"
    runtime = ROOT / "refs/TinyML/upstream-2026.1/tinyml_hello_world/Ti60F225_tinyml_hello_world/embedded_sw/SapphireSoc"
    sdk = desktop / "Work/FPGA Contest/env/RISCV-IDE"
    subprocess.run([sys.executable, str(FPGA / "tools/build_cnn_static.py"), "--vendor-workspace", str(vendor), "--runtime-workspace", str(runtime), "--sdk", str(sdk), "--bundle", str(ROOT / "artifacts/v0.3-board-bundle"), "--output", str(FPGA / "artifacts/cnn-static-notebook")], cwd=FPGA, check=True)
else:
    print("默认回放模式；本次未启动重新编译。")

默认回放模式；本次未启动重新编译。


## 5. 板端需要怎样的证据

UART 日志需记录相同模型 SHA、有效 arena 占用、三组实际 logits 及周期；每组 `max_abs=0` 且最后 `exact_vectors=3/3 PASS` 才通过逐字节检查。cycle 高低位合并为 `(hi << 32) | lo`，除以实际 CPU Hz 才是秒。没有实测日志时不计算或宣称 FPS。

当前静态 BSP 内存与视频帧槽冲突；接入前阅读 [CI、DMA 与缓存约定](../docs/04_deployment/riscv_custom_instruction.md)。

## 语法简表

- `Path / name`：组合路径，不依赖字符串反斜杠转义。
- `json.loads(...)`：将实验报告还原为字典。
- `assert`：核对数据一致性，失败会立即指出问题。
- `subprocess.run([...], check=True)`：参数列表调用工具，失败抛错；路径有空格也无需自己拼 shell 命令。